<a href="https://colab.research.google.com/github/jaeheechillin3/pytorch_projects/blob/main/basic_implementations/simple_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class-conditioned MNIST diffusion

Train a DDPM to generate requested digits using a learned label embedding added to the timestep embedding. Images are padded from 28×28 to 32×32. The default trains on all 60,000 images for 100 epochs; set `TRAIN_SAMPLES = 10_000` and `EPOCHS = 30` for a shorter experiment.

Run the cells in order on a GPU. This model requires fresh training; the earlier unconditional KMNIST checkpoint is incompatible. Samples are conditioned on digits 0–9, but generation can still make mistakes.

Based on [Diffusion models from scratch in PyTorch](https://www.youtube.com/watch?v=a4Yfz2FxXiY) and [DDPM](https://arxiv.org/abs/2006.11239), with class conditioning added here.

## Imports

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

## Configuration

In [ ]:
DATASET = "mnist"
NUM_CLASSES = 10
DATA_ROOT = "./data"
IMAGE_SIZE = 32
IMAGE_CHANNELS = 1
TRAIN_SAMPLES = None     # All 60,000 images; use 10_000 for a shorter run.
BASE_CHANNELS = 32
BATCH_SIZE = 32
TIMESTEPS = 1000
EPOCHS = 100
LEARNING_RATE = 1e-4
SAMPLE_EVERY = 5
OUTPUT_DIR = "./mnist_conditional_diffusion_outputs"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Helper Functions

In [ ]:
def load_dataset():
    """Download MNIST and optionally select a reproducible training subset."""
    transform = transforms.Compose([
        # Padding preserves character strokes and yields dimensions divisible by 16.
        transforms.Pad(2, fill=0),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,), std=(0.5,)),
    ])
    dataset = datasets.MNIST(
        DATA_ROOT, train=True, transform=transform, download=True
    )
    if TRAIN_SAMPLES is None:
        return dataset
    if not 1 <= TRAIN_SAMPLES <= len(dataset):
        raise ValueError(f"TRAIN_SAMPLES must be between 1 and {len(dataset)}.")
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(len(dataset), generator=generator)[:TRAIN_SAMPLES]
    return Subset(dataset, indices.tolist())

In [ ]:
def show_images(images, title=None, save_path=None, labels=None):
    """Display a CHW image or BCHW batch normalized to [-1, 1]."""
    if images.ndim == 3:
        images = images.unsqueeze(0)
    images = (images.detach().cpu().float() / 2 + 0.5).clamp(0, 1)
    count = len(images)
    if labels is not None:
        labels = torch.as_tensor(labels).detach().cpu().tolist()
        if len(labels) != count:
            raise ValueError("Provide one label per image.")
    cols = min(5, count)
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for index, (axis, image) in enumerate(zip(axes.flat, images)):
        if labels is not None:
            axis.set_title(f"Requested digit: {labels[index]}")
        if image.shape[0] == 1:
            axis.imshow(image[0], cmap="gray", vmin=0, vmax=1)
        else:
            axis.imshow(image.permute(1, 2, 0))
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Model

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        if dim < 4 or dim % 2:
            raise ValueError("Time embedding dimension must be even and >= 4.")
        half_dim = dim // 2
        frequencies = torch.exp(
            -math.log(10_000) * torch.arange(half_dim) / (half_dim - 1)
        )
        self.register_buffer("frequencies", frequencies)

    def forward(self, time):
        angles = time.float()[:, None] * self.frequencies[None, :]
        return torch.cat((angles.sin(), angles.cos()), dim=-1)

In [ ]:
class Block(nn.Module):
    """Two convolutions with timestep conditioning, then down/up-sampling."""
    def __init__(self, in_channels, out_channels, time_dim, up=False):
        super().__init__()
        input_channels = 2 * in_channels if up else in_channels
        self.conv1 = nn.Conv2d(input_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        # GroupNorm does not depend on batch statistics during sampling.
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_channels))
        if up:
            self.resample = nn.ConvTranspose2d(out_channels, out_channels, 4, 2, 1)
        else:
            self.resample = nn.Conv2d(out_channels, out_channels, 4, 2, 1)

    def forward(self, x, time_embedding):
        x = F.silu(self.norm1(self.conv1(x)))
        x = x + self.time_mlp(time_embedding)[:, :, None, None]
        x = F.silu(self.norm2(self.conv2(x)))
        return self.resample(x)

In [ ]:
class SimpleUnet(nn.Module):
    def __init__(
        self, image_channels=1, base_channels=32, time_dim=32, num_classes=10
    ):
        super().__init__()
        if base_channels < 8 or base_channels % 8:
            raise ValueError("base_channels must be a positive multiple of 8.")
        channels = [base_channels * factor for factor in (1, 2, 4, 8, 16)]
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        self.label_embedding = nn.Embedding(num_classes, time_dim)
        self.conv0 = nn.Conv2d(image_channels, channels[0], 3, padding=1)
        self.downs = nn.ModuleList([
            Block(channels[i], channels[i + 1], time_dim) for i in range(4)
        ])
        self.ups = nn.ModuleList([
            Block(channels[i], channels[i - 1], time_dim, up=True)
            for i in range(4, 0, -1)
        ])
        self.output = nn.Conv2d(channels[0], image_channels, 1)

    def forward(self, x, time, labels):
        if x.shape[-2] % 16 or x.shape[-1] % 16:
            raise ValueError("Image height and width must be divisible by 16.")
        # The same digit condition is supplied to every down/up block.
        time_embedding = self.time_mlp(time) + self.label_embedding(labels)
        x = self.conv0(x)
        skips = []
        for down in self.downs:
            x = down(x, time_embedding)
            skips.append(x)
        for up in self.ups:
            x = torch.cat((x, skips.pop()), dim=1)
            x = up(x, time_embedding)
        return self.output(x)

In [ ]:
class Diffusion(nn.Module):
    """DDPM schedule and forward/reverse processes; buffers follow .to(device)."""
    def __init__(self, timesteps=1000):
        super().__init__()
        if timesteps < 2:
            raise ValueError("timesteps must be >= 2.")
        self.timesteps = timesteps
        betas = torch.linspace(1e-4, 0.02, timesteps)
        alphas = 1 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        alpha_bar_previous = F.pad(alpha_bar[:-1], (1, 0), value=1.0)
        coefficients = {
            "sqrt_alpha_bar": alpha_bar.sqrt(),
            "sqrt_one_minus_alpha_bar": (1 - alpha_bar).sqrt(),
            "posterior_mean_x0": betas * alpha_bar_previous.sqrt() / (1 - alpha_bar),
            "posterior_mean_xt": (
                (1 - alpha_bar_previous) * alphas.sqrt() / (1 - alpha_bar)
            ),
            "posterior_variance": (
                betas * (1 - alpha_bar_previous) / (1 - alpha_bar)
            ),
        }
        for name, values in coefficients.items():
            self.register_buffer(name, values)

    @staticmethod
    def extract(values, time, x):
        return values.gather(0, time).reshape(-1, *([1] * (x.ndim - 1)))

    def forward_diffusion_sample(self, clean_images, time, noise=None):
        if noise is None:
            noise = torch.randn_like(clean_images)
        noisy_images = (
            self.extract(self.sqrt_alpha_bar, time, clean_images) * clean_images
            + self.extract(self.sqrt_one_minus_alpha_bar, time, clean_images) * noise
        )
        return noisy_images, noise

    def loss(self, model, clean_images, time, labels):
        noisy_images, noise = self.forward_diffusion_sample(clean_images, time)
        return F.mse_loss(model(noisy_images, time, labels), noise)

    @torch.no_grad()
    def sample_timestep(self, model, x, time, labels):
        predicted_noise = model(x, time, labels)
        # Clip the estimated clean image, NOT the noisy intermediate state.
        predicted_x0 = (
            (x - self.extract(self.sqrt_one_minus_alpha_bar, time, x) * predicted_noise)
            / self.extract(self.sqrt_alpha_bar, time, x)
        ).clamp(-1, 1)
        mean = (
            self.extract(self.posterior_mean_x0, time, x) * predicted_x0
            + self.extract(self.posterior_mean_xt, time, x) * x
        )
        variance = self.extract(self.posterior_variance, time, x)
        nonzero = (time > 0).reshape(-1, *([1] * (x.ndim - 1)))
        return mean + nonzero * variance.sqrt() * torch.randn_like(x)

    @torch.no_grad()
    def sample(self, model, shape, labels, initial_noise=None):
        labels = torch.as_tensor(labels, device=self.sqrt_alpha_bar.device)
        if labels.dtype != torch.long:
            raise ValueError("labels must contain integer digit IDs (torch.long).")
        if labels.shape != (shape[0],):
            raise ValueError("Provide one digit label per generated image.")
        if ((labels < 0) | (labels >= model.label_embedding.num_embeddings)).any():
            raise ValueError("Digit labels must be in the range 0–9.")
        was_training = model.training
        model.eval()
        try:
            device = self.sqrt_alpha_bar.device
            if initial_noise is None:
                x = torch.randn(shape, device=device)
            else:
                x = initial_noise.to(device).clone()
            if tuple(x.shape) != tuple(shape):
                raise ValueError("initial_noise must match the requested shape.")
            for step in reversed(range(self.timesteps)):
                time = torch.full((shape[0],), step, device=device, dtype=torch.long)
                x = self.sample_timestep(model, x, time, labels)
            return x
        finally:
            model.train(was_training)

## Training

In [ ]:
def main():
    torch.manual_seed(42)
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset = load_dataset()
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )
    model = SimpleUnet(
        IMAGE_CHANNELS, BASE_CHANNELS, num_classes=NUM_CLASSES
    ).to(DEVICE)
    diffusion = Diffusion(TIMESTEPS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = []
    sample_labels = torch.arange(NUM_CLASSES, device=DEVICE)
    sample_shape = (NUM_CLASSES, IMAGE_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    fixed_noise = torch.randn(sample_shape, device=DEVICE)
    print(f"Device: {DEVICE} | Training images: {len(dataset):,}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    show_images(torch.stack([dataset[i][0] for i in range(5)]), "Training examples")
    clean_image = dataset[0][0].unsqueeze(0).to(DEVICE)
    times = torch.linspace(0, TIMESTEPS - 1, 10, device=DEVICE).long()
    # Reuse one noise realization so the forward-process comparison is clearer.
    noise = torch.randn_like(clean_image).expand(10, -1, -1, -1)
    noisy, _ = diffusion.forward_diffusion_sample(
        clean_image.expand(10, -1, -1, -1), times, noise
    )
    show_images(noisy, "Forward diffusion: increasing noise (left to right, top to bottom)")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_images = 0
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, dtype=torch.long, non_blocking=True)
            time = torch.randint(TIMESTEPS, (images.size(0),), device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = diffusion.loss(model, images, time, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            total_images += images.size(0)
        history.append(total_loss / total_images)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Noise MSE: {history[-1]:.5f}", flush=True)

        # Save before sampling so training progress survives plotting failures.
        torch.save({
            "model": model.state_dict(), "diffusion": diffusion.state_dict(),
            "optimizer": optimizer.state_dict(), "epoch": epoch, "history": history,
            "config": {
                "dataset": DATASET, "num_classes": NUM_CLASSES,
                "conditional": True, "image_size": IMAGE_SIZE,
                "image_channels": IMAGE_CHANNELS, "base_channels": BASE_CHANNELS,
                "timesteps": TIMESTEPS, "train_samples": TRAIN_SAMPLES,
            },
        }, output_dir / "checkpoint.pt")
        if epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
            samples = diffusion.sample(
                model, sample_shape, sample_labels, initial_noise=fixed_noise
            )
            show_images(
                samples,
                f"Conditional MNIST samples: epoch {epoch}",
                output_dir / f"samples_{epoch:03d}.png",
                labels=sample_labels,
            )

    fig, axis = plt.subplots(figsize=(7, 4))
    axis.plot(range(1, EPOCHS + 1), history)
    axis.set(xlabel="Epoch", ylabel="Noise-prediction MSE", title="Training loss")
    axis.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_dir / "training_loss.png")
    plt.show()
    plt.close(fig)
    return model, diffusion, history

In [ ]:
if __name__ == "__main__":
    model, diffusion, history = main()

In [ ]:
# Request one sample for each digit. Repeat a label to generate more of that digit.
requested_labels = torch.arange(NUM_CLASSES, device=DEVICE)
samples = diffusion.sample(
    model,
    shape=(len(requested_labels), IMAGE_CHANNELS, IMAGE_SIZE, IMAGE_SIZE),
    labels=requested_labels,
)

show_images(
    samples,
    title="New generated MNIST digits",
    labels=requested_labels,
)
